# Döntési fa — Vizualizáció

**Felelős:** Kovács Kornél

Ez a notebook **csak vizualizál**. A `03_decision_tree.ipynb` által Drive-ra mentett, **verziózott** artefaktokat olvassa be (`{VERSION}` placeholder a 2. szekcióban):
- `decision_tree_tuned_{VERSION}.joblib`
- `decision_tree_predictions_test_{VERSION}.npy`
- `decision_tree_validation_curve_max_depth_{VERSION}.npz`
- `decision_tree_learning_curve_{VERSION}.npz`

+ a `data_io.load_v1_data(version=VERSION)`-ből a `y_train`, `y_test`, `X_test`, `feature_names`.

A mentett PNG-k is verziózott névvel kerülnek `FIGURES_DIR`-be (pl. `dt_predicted_vs_actual_v1.png`), így a v0/vKNN/v1 plot-ok egymás mellett megférnek. Sklearn fitmentes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Slityak/gepitan-seoul-bike-trip/blob/main/notebooks/03_decision_tree_visualize.ipynb)

## 1. Setup (Colab + lokális kompatibilis)

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
BRANCH = 'feature/cuccos'  # mergelés után állítsd vissza 'main'-re

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    if not os.path.exists('/content/gepitan-beadando'):
        !git clone --branch {BRANCH} https://github.com/Slityak/gepitan-seoul-bike-trip.git /content/gepitan-beadando

    os.chdir('/content/gepitan-beadando')
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
    !pip install -q -r requirements.txt
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')
print(f'In Colab: {IN_COLAB}')
print(f'Branch: {BRANCH}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import joblib

from src.config import MODELS_DIR, FIGURES_DIR, ensure_dirs
from src.data_io import load_v1_data
from src.visualization import (
    plot_decision_tree_top,
    plot_error_by_feature_quantile,
    plot_feature_importance,
    plot_learning_curve,
    plot_predicted_vs_actual,
    plot_residuals,
    plot_target_distribution,
    plot_validation_curve,
)

ensure_dirs()

## 2. Adatok és mentett artefaktok betöltése

In [ ]:
VERSION = 'v0'  # 'v0' (200k sample) | 'vKNN' (500k) | 'v1' (teljes 9.6M)
DT_LABEL = f'Decision Tree (tuned, {VERSION})'

X_train, X_test, y_train, y_test, feature_names = load_v1_data(version=VERSION)

best_model = joblib.load(MODELS_DIR / f'decision_tree_tuned_{VERSION}.joblib')
y_pred_tuned = np.load(MODELS_DIR / f'decision_tree_predictions_test_{VERSION}.npy')
vc = np.load(MODELS_DIR / f'decision_tree_validation_curve_max_depth_{VERSION}.npz')
lc = np.load(MODELS_DIR / f'decision_tree_learning_curve_{VERSION}.npz')

print(f'Verzió: {VERSION}')
print(f'best_model params: {best_model.get_params()}')
print(f'y_pred_tuned shape: {y_pred_tuned.shape}')
print(f'validation curve: {len(vc["param_range_labels"])} max_depth érték')
print(f'learning curve:   {len(lc["sizes"])} train size pont')

## 3. Predicted vs actual

In [ ]:
plot_predicted_vs_actual(
    y_test, y_pred_tuned,
    model_name=DT_LABEL,
    save_path=f'dt_predicted_vs_actual_{VERSION}.png',
)
plt.show()

## 4. Residuals

In [ ]:
plot_residuals(
    y_test, y_pred_tuned,
    model_name=DT_LABEL,
    save_path=f'dt_residuals_{VERSION}.png',
)
plt.show()

## 5. Feature importance

In [ ]:
plot_feature_importance(
    feature_names=feature_names,
    importances=best_model.feature_importances_,
    model_name=DT_LABEL,
    top_n=20,
    save_path=f'dt_feature_importance_{VERSION}.png',
)
plt.show()

## 6. Target eloszlás

In [ ]:
plot_target_distribution(y_train, save_path=f'dt_target_distribution_{VERSION}.png')
plt.show()

## 7. Validation curve (max_depth)

In [ ]:
plot_validation_curve(
    param_range_labels=list(vc['param_range_labels']),
    train_mae=vc['train_mae'],
    test_mae=vc['test_mae'],
    param_name='max_depth',
    model_name=f'Decision Tree ({VERSION})',
    save_path=f'dt_validation_curve_max_depth_{VERSION}.png',
)
plt.show()

## 8. Learning curve

In [ ]:
plot_learning_curve(
    sizes=lc['sizes'],
    train_mae=lc['train_mae'],
    test_mae=lc['test_mae'],
    model_name=DT_LABEL,
    save_path=f'dt_learning_curve_{VERSION}.png',
)
plt.show()

## 9. Hiba feature szerint

In [ ]:
hav_idx = feature_names.index('Haversine')
plot_error_by_feature_quantile(
    y_test, y_pred_tuned, X_test[:, hav_idx],
    feature_name='Haversine',
    n_bins=10,
    model_name=DT_LABEL,
    save_path=f'dt_error_by_haversine_{VERSION}.png',
)
plt.show()

phour_idx = feature_names.index('Phour')
plot_error_by_feature_quantile(
    y_test, y_pred_tuned, X_test[:, phour_idx],
    feature_name='Phour',
    n_bins=10,
    model_name=DT_LABEL,
    save_path=f'dt_error_by_phour_{VERSION}.png',
)
plt.show()

## 10. Sub-tree (felső 3 szint)

In [ ]:
plot_decision_tree_top(
    best_model,
    feature_names=feature_names,
    max_depth=3,
    model_name=DT_LABEL,
    save_path=f'dt_subtree_top3_{VERSION}.png',
)
plt.show()